In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical Anomaly detection
from scipy.stats import median_abs_deviation

# Residual-based detection
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Unsupervised Anomaly detection
from sklearn.ensemble import IsolationForest

# Model persistence
import joblib

import warnings
warnings.filterwarnings("ignore")

In [2]:
engg_df = pd.read_csv("data/engineered_data.csv")

In [4]:
# Verifying the engineered dataset
print("Dataset shape:", engg_df.shape)

display(engg_df.head())

print("\nColumn names:")
print(engg_df.columns.tolist())

print("\nData types:")
print(engg_df.dtypes)

Dataset shape: (240, 23)


,timestamp_id,asset_id,irradiance_wm2,wind_speed_ms,temperature_c,vibration_mm_s,power_mw,asset_type,hour_sin,hour_cos,...,vibration_mm_s_rolling_median,vibration_mm_s_rolling_mad,vibration_mm_s_modified_zscore,irradiance_wm2_rolling_median,irradiance_wm2_rolling_mad,irradiance_wm2_modified_zscore,wind_speed_ms_rolling_median,wind_speed_ms_rolling_mad,wind_speed_ms_modified_zscore,power_change_mw
0,1970-01-01 00:00:00.000000000,SOLAR_INV_01,0.0,6.57,26.26,1.37,0.0,solar,0.000000,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1970-01-01 00:00:00.000000001,SOLAR_INV_01,0.0,6.72,24.89,1.49,0.0,solar,0.258819,0.965926,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,1970-01-01 00:00:00.000000002,SOLAR_INV_01,0.0,6.26,23.85,1.25,0.0,solar,0.500000,0.866025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,1970-01-01 00:00:00.000000003,SOLAR_INV_01,0.0,7.24,24.45,1.29,0.0,solar,0.707107,0.707107,...,1.37,NaN,NaN,0.0,NaN,NaN,6.570,NaN,NaN,0.0
4,1970-01-01 00:00:00.000000004,SOLAR_INV_01,0.0,7.39,26.45,1.50,0.0,solar,0.866025,0.500000,...,1.33,NaN,NaN,0.0,NaN,NaN,6.645,NaN,NaN,0.0



Column names:
['timestamp_id', 'asset_id', 'irradiance_wm2', 'wind_speed_ms', 'temperature_c', 'vibration_mm_s', 'power_mw', 'asset_type', 'hour_sin', 'hour_cos', 'power_mw_rolling_median', 'power_mw_rolling_mad', 'power_mw_modified_zscore', 'vibration_mm_s_rolling_median', 'vibration_mm_s_rolling_mad', 'vibration_mm_s_modified_zscore', 'irradiance_wm2_rolling_median', 'irradiance_wm2_rolling_mad', 'irradiance_wm2_modified_zscore', 'wind_speed_ms_rolling_median', 'wind_speed_ms_rolling_mad', 'wind_speed_ms_modified_zscore', 'power_change_mw']

Data types:
timestamp_id                       object
asset_id                           object
irradiance_wm2                    float64
wind_speed_ms                     float64
temperature_c                     float64
vibration_mm_s                    float64
power_mw                          float64
asset_type                         object
hour_sin                          float64
hour_cos                          float64
power_mw_rolling_

In [6]:
# Verifying raw sensor completeness
sensor_cols = [
    "irradiance_wm2",
    "wind_speed_ms",
    "temperature_c",
    "vibration_mm_s",
    "power_mw"
]

print("Missing raw sensor values:")
print(engg_df[sensor_cols].isnull().sum())

# Inspect missing values in engineered features
engineered_cols = [
    "hour_sin",
    "hour_cos",
    "power_mw_rolling_median",
    "power_mw_rolling_mad"
]

existing_cols = [
    col for col in engineered_cols
    if col in engg_df.columns
]

print("\nMissing engineered feature values:")
print(engg_df[existing_cols].isnull().sum())

Missing raw sensor values:
irradiance_wm2    0
wind_speed_ms     0
temperature_c     0
vibration_mm_s    0
power_mw          0
dtype: int64

Missing engineered feature values:
hour_sin                    0
hour_cos                    0
power_mw_rolling_median     6
power_mw_rolling_mad       12
dtype: int64


# Staistical Rule Based Detection

In [ ]:
# Inspecting sensor ranges separately for each asset type

sensor_cols = [
    "irradiance_wm2",
    "wind_speed_ms",
    "temperature_c",
    "vibration_mm_s",
    "power_mw"
]

for asset_type, group in engg_df.groupby("asset_type"):
    print(f"\nAsset type: {asset_type}")
    display(group[sensor_cols].describe().T)


Asset type: solar


,count,mean,std,min,25%,50%,75%,max
irradiance_wm2,120.0,253.191667,310.958371,0.00,0.000,0.000,565.700,800.00
wind_speed_ms,120.0,6.295083,1.606883,3.16,4.925,6.535,7.655,8.99
temperature_c,120.0,32.354833,6.465789,22.56,26.395,32.035,37.730,53.48
vibration_mm_s,120.0,1.426917,0.401650,1.20,1.290,1.375,1.440,3.98
power_mw,120.0,0.481000,0.590785,0.00,0.000,0.000,1.075,1.52



Asset type: wind


,count,mean,std,min,25%,50%,75%,max
irradiance_wm2,120.0,0.000000,0.000000,0.000,0.00000,0.000,0.0000,0.000
wind_speed_ms,120.0,5.843583,1.528620,3.170,4.53750,5.875,7.0150,8.820
temperature_c,120.0,32.269750,6.249425,22.350,26.26750,32.635,38.0550,46.880
vibration_mm_s,120.0,1.374417,0.324987,1.200,1.24750,1.340,1.4100,3.820
power_mw,120.0,1.269733,0.691727,0.077,0.69175,1.249,1.8085,2.618


Observations from Descriptive Staistics

1. Solar: Irradiance and power both have a median of zero, so zero power can be normal under low-light conditions. We should not flag it automatically.
2. Wind: Irradiance is consistently zero, as expected for this dataset's wind assets. Wind power ranges from 0.077 to 2.618 MW.
3. Both asset types: Vibration has unusually high maximum values (3.98 mm/s for solar and 3.82 mm/s for wind), while their 75th percentiles are only 1.44 and 1.41 mm/s, respectively. These are candidates for investigation but they are not confirmed faults.
4. Temperature: The maximum solar temperature of 53.48°C deserves inspection, but we cannot declare it anomalous without a justified operating limit.

In [8]:
# Inspect potentially unusual vibration and temperature readings

In [9]:
display(
    engg_df.loc[
        engg_df["vibration_mm_s"] > 3.5,
        [
            "timestamp_id",
            "asset_id",
            "asset_type",
            "temperature_c",
            "vibration_mm_s",
            "power_mw"
        ]
    ].sort_values("vibration_mm_s", ascending=False)
)

# Inspect the highest solar temperature reading(s)
display(
    engg_df.loc[
        (engg_df["asset_type"] == "solar")
        & (engg_df["temperature_c"] >= 50),
        [
            "timestamp_id",
            "asset_id",
            "temperature_c",
            "irradiance_wm2",
            "power_mw",
            "vibration_mm_s"
        ]
    ]
)

,timestamp_id,asset_id,asset_type,temperature_c,vibration_mm_s,power_mw
68,1970-01-01 00:00:00.000000068,SOLAR_INV_01,solar,48.55,3.98,0.000
67,1970-01-01 00:00:00.000000067,SOLAR_INV_01,solar,53.48,3.86,0.000
174,1970-01-01 00:00:00.000000174,WTG_07,wind,46.88,3.82,0.711
69,1970-01-01 00:00:00.000000069,SOLAR_INV_01,solar,46.04,3.77,0.000
173,1970-01-01 00:00:00.000000173,WTG_07,wind,45.43,3.72,0.726


,timestamp_id,asset_id,temperature_c,irradiance_wm2,power_mw,vibration_mm_s
67,1970-01-01 00:00:00.000000067,SOLAR_INV_01,53.48,0.0,0.0,3.86


Observations- 

1. Solar: Three consecutive observations (IDs 67–69) have high vibration (3.77–3.98 mm/s), zero power, and elevated temperatures. This may have a unusual operating condition, but we cannot confirm a fault from these readings alone.

2. Wind: Two consecutive observations (IDs 173–174) have high vibration (3.72–3.82 mm/s), with power around 0.71–0.73 MW. These also are to be investigated.

In [ ]:
# Potential Anamoly rules

# Rule 1: Unusually high vibration
engg_df["high_vibration_flag"] = (
    engg_df["vibration_mm_s"] > 3.5
)

# Rule 2: High temperature requiring investigation , 45 degree is screening flag not confirmed
engg_df["high_temperature_flag"] = (
    engg_df["temperature_c"] > 45
)

# Inspect observations flagged by either rule
flagged_df = engg_df.loc[
    engg_df["high_vibration_flag"]
    | engg_df["high_temperature_flag"],
    [
        "timestamp_id",
        "asset_id",
        "asset_type",
        "temperature_c",
        "vibration_mm_s",
        "power_mw",
        "high_vibration_flag",
        "high_temperature_flag"
    ]
]

display(flagged_df)

,timestamp_id,asset_id,asset_type,temperature_c,vibration_mm_s,power_mw,high_vibration_flag,high_temperature_flag
67,1970-01-01 00:00:00.000000067,SOLAR_INV_01,solar,53.48,3.86,0.000,True,True
68,1970-01-01 00:00:00.000000068,SOLAR_INV_01,solar,48.55,3.98,0.000,True,True
69,1970-01-01 00:00:00.000000069,SOLAR_INV_01,solar,46.04,3.77,0.000,True,True
173,1970-01-01 00:00:00.000000173,WTG_07,wind,45.43,3.72,0.726,True,True
174,1970-01-01 00:00:00.000000174,WTG_07,wind,46.88,3.82,0.711,True,True


In [12]:
# Summarizing rule-based detection results

print("Total observations:", len(engg_df))

print(
    "High vibration:",
    engg_df["high_vibration_flag"].sum()
)

print(
    "High temperature:",
    engg_df["high_temperature_flag"].sum()
)

print(
    "Both rules triggered:",
    (
        engg_df["high_vibration_flag"]
        & engg_df["high_temperature_flag"]
    ).sum()
)

print(
    "Flagged by either rule:",
    (
        engg_df["high_vibration_flag"]
        | engg_df["high_temperature_flag"]
    ).sum()
)

Total observations: 240
High vibration: 5
High temperature: 5
Both rules triggered: 5
Flagged by either rule: 5


In [15]:
# Summarizing flagged observations by asset type

flagged_summary = (
    engg_df.loc[
        engg_df["high_vibration_flag"]
        | engg_df["high_temperature_flag"]
    ]
    .groupby("asset_type")
    .agg(
        flagged_observations=("asset_id", "size"),
        affected_assets=("asset_id", "nunique"),
        mean_temperature=("temperature_c", "mean"),
        mean_vibration=("vibration_mm_s", "mean"),
        mean_power=("power_mw", "mean")
    )
    .reset_index()
)

display(flagged_summary)

,asset_type,flagged_observations,affected_assets,mean_temperature,mean_vibration,mean_power
0,solar,3,1,49.356667,3.87,0.0000
1,wind,2,1,46.155000,3.77,0.7185


*** Method 1 - Findings

- **Total observations:** 240
- **Flagged observations:** 5 (2.08%)
- **Asset types:** Solar and wind
- All five observations exceeded both the vibration threshold (3.5 mm/s) and temperature screening threshold (45°C).
- The solar observations had zero power output, while the wind observations retained power output.

**Note:** These are candidate anomalies, not confirmed faults. Detailed evaluation and comparison will be performed after completing the detection methods.

# Rolling Median and MAD-based anamoly method

In [16]:
# Sorting observations chronologically by sequence ID within each asset
engg_df.sort_values(
    ["asset_id", "timestamp_id"],
    inplace=True
)

# Rolling window configuration
window_size = 12
min_periods = 6

# Calculating the historical rolling median of power
engg_df["method2_rolling_median"] = (
    engg_df.groupby("asset_id")["power_mw"]
    .transform(
        lambda s: s.shift(1).rolling(
            window=window_size,
            min_periods=min_periods
        ).median()
    )
)

# Calculating the historical rolling MAD of power
engg_df["method2_rolling_mad"] = (
    engg_df.groupby("asset_id")["power_mw"]
    .transform(
        lambda s: s.shift(1).rolling(
            window=window_size,
            min_periods=min_periods
        ).apply(
            lambda x: np.median(
                np.abs(x - np.median(x))
            ),
            raw=True
        )
    )
)

# Inspect the calculated features
display(
    engg_df[
        [
            "timestamp_id",
            "asset_id",
            "asset_type",
            "power_mw",
            "method2_rolling_median",
            "method2_rolling_mad"
        ]
    ].head(20)
)

,timestamp_id,asset_id,asset_type,power_mw,method2_rolling_median,method2_rolling_mad
0,1970-01-01 00:00:00.000000000,SOLAR_INV_01,solar,0.000,NaN,NaN
1,1970-01-01 00:00:00.000000001,SOLAR_INV_01,solar,0.000,NaN,NaN
2,1970-01-01 00:00:00.000000002,SOLAR_INV_01,solar,0.000,NaN,NaN
3,1970-01-01 00:00:00.000000003,SOLAR_INV_01,solar,0.000,NaN,NaN
4,1970-01-01 00:00:00.000000004,SOLAR_INV_01,solar,0.000,NaN,NaN
5,1970-01-01 00:00:00.000000005,SOLAR_INV_01,solar,0.000,NaN,NaN
6,1970-01-01 00:00:00.000000006,SOLAR_INV_01,solar,0.000,0.0000,NaN
7,1970-01-01 00:00:00.000000007,SOLAR_INV_01,solar,0.393,0.0000,NaN
8,1970-01-01 00:00:00.000000008,SOLAR_INV_01,solar,0.760,0.0000,NaN
9,1970-01-01 00:00:00.000000009,SOLAR_INV_01,solar,1.075,0.0000,NaN


In [17]:
# Calculating the rolling MAD-based deviation score
mad = engg_df["method2_rolling_mad"]

engg_df["method2_deviation_score"] = np.where(
    mad > 0,
    0.6745
    * (
        engg_df["power_mw"]
        - engg_df["method2_rolling_median"]
    )
    / mad,
    np.nan
)

# Flagging potential anomalies using the initial threshold
threshold = 3.5

engg_df["method2_anomaly_flag"] = (
    engg_df["method2_deviation_score"].abs() > threshold
)

# Missing scores cannot be evaluated; mark them as unflagged
engg_df["method2_anomaly_flag"] = (
    engg_df["method2_anomaly_flag"].fillna(False)
)

# Inspect flagged observations
display(
    engg_df.loc[
        engg_df["method2_anomaly_flag"],
        [
            "timestamp_id",
            "asset_id",
            "asset_type",
            "power_mw",
            "method2_rolling_median",
            "method2_rolling_mad",
            "method2_deviation_score"
        ]
    ]
)

print(
    "Potential anomalies detected:",
    engg_df["method2_anomaly_flag"].sum()
)

,timestamp_id,asset_id,asset_type,power_mw,method2_rolling_median,method2_rolling_mad,method2_deviation_score
13,1970-01-01 00:00:00.000000013,SOLAR_INV_01,solar,1.468,0.1965,0.1965,4.364513
37,1970-01-01 00:00:00.000000037,SOLAR_INV_01,solar,1.468,0.1965,0.1965,4.364513
61,1970-01-01 00:00:00.000000061,SOLAR_INV_01,solar,1.468,0.1965,0.1965,4.364513
85,1970-01-01 00:00:00.000000085,SOLAR_INV_01,solar,1.468,0.1965,0.1965,4.364513
109,1970-01-01 00:00:00.000000109,SOLAR_INV_01,solar,1.468,0.1965,0.1965,4.364513
173,1970-01-01 00:00:00.000000173,WTG_07,wind,0.726,2.1790,0.1865,-5.254952
174,1970-01-01 00:00:00.000000174,WTG_07,wind,0.711,2.1445,0.1885,-5.129420
175,1970-01-01 00:00:00.000000175,WTG_07,wind,1.105,2.1445,0.1885,-3.719590
198,1970-01-01 00:00:00.000000198,WTG_07,wind,1.140,0.3830,0.1275,4.004678
199,1970-01-01 00:00:00.000000199,WTG_07,wind,1.521,0.4535,0.1615,4.458382


Potential anomalies detected: 11


In [19]:
# Summarizing this  Method 2 detections by asset type
print(
    engg_df.loc[engg_df["method2_anomaly_flag"]]
    .groupby("asset_type")
    .size()
    .rename("flagged_observations")
)

# Inspecting flagged observations
display(
    engg_df.loc[
        engg_df["method2_anomaly_flag"],
        [
            "timestamp_id",
            "asset_id",
            "asset_type",
            "power_mw",
            "method2_rolling_median",
            "method2_rolling_mad",
            "method2_deviation_score"
        ]
    ].sort_values(
        ["asset_id", "timestamp_id"]
    )
)

asset_type
solar    5
wind     6
Name: flagged_observations, dtype: int64


,timestamp_id,asset_id,asset_type,power_mw,method2_rolling_median,method2_rolling_mad,method2_deviation_score
13,1970-01-01 00:00:00.000000013,SOLAR_INV_01,solar,1.468,0.1965,0.1965,4.364513
37,1970-01-01 00:00:00.000000037,SOLAR_INV_01,solar,1.468,0.1965,0.1965,4.364513
61,1970-01-01 00:00:00.000000061,SOLAR_INV_01,solar,1.468,0.1965,0.1965,4.364513
85,1970-01-01 00:00:00.000000085,SOLAR_INV_01,solar,1.468,0.1965,0.1965,4.364513
109,1970-01-01 00:00:00.000000109,SOLAR_INV_01,solar,1.468,0.1965,0.1965,4.364513
173,1970-01-01 00:00:00.000000173,WTG_07,wind,0.726,2.1790,0.1865,-5.254952
174,1970-01-01 00:00:00.000000174,WTG_07,wind,0.711,2.1445,0.1885,-5.129420
175,1970-01-01 00:00:00.000000175,WTG_07,wind,1.105,2.1445,0.1885,-3.719590
198,1970-01-01 00:00:00.000000198,WTG_07,wind,1.140,0.3830,0.1275,4.004678
199,1970-01-01 00:00:00.000000199,WTG_07,wind,1.521,0.4535,0.1615,4.458382


In [20]:
# Inspecting flagged observations with relevant sensor context

display(
    engg_df.loc[
        engg_df["method2_anomaly_flag"],
        [
            "timestamp_id",
            "asset_id",
            "asset_type",
            "irradiance_wm2",
            "wind_speed_ms",
            "temperature_c",
            "vibration_mm_s",
            "power_mw",
            "method2_rolling_median",
            "method2_rolling_mad",
            "method2_deviation_score"
        ]
    ].sort_values(["asset_id", "timestamp_id"])
)

,timestamp_id,asset_id,asset_type,irradiance_wm2,wind_speed_ms,temperature_c,vibration_mm_s,power_mw,method2_rolling_median,method2_rolling_mad,method2_deviation_score
13,1970-01-01 00:00:00.000000013,SOLAR_INV_01,solar,772.7,8.99,40.74,1.48,1.468,0.1965,0.1965,4.364513
37,1970-01-01 00:00:00.000000037,SOLAR_INV_01,solar,772.7,4.42,40.29,1.39,1.468,0.1965,0.1965,4.364513
61,1970-01-01 00:00:00.000000061,SOLAR_INV_01,solar,772.7,7.44,38.12,1.39,1.468,0.1965,0.1965,4.364513
85,1970-01-01 00:00:00.000000085,SOLAR_INV_01,solar,772.7,3.16,40.28,1.30,1.468,0.1965,0.1965,4.364513
109,1970-01-01 00:00:00.000000109,SOLAR_INV_01,solar,772.7,8.52,41.71,1.23,1.468,0.1965,0.1965,4.364513
173,1970-01-01 00:00:00.000000173,WTG_07,wind,0.0,5.93,45.43,3.72,0.726,2.1790,0.1865,-5.254952
174,1970-01-01 00:00:00.000000174,WTG_07,wind,0.0,5.87,46.88,3.82,0.711,2.1445,0.1885,-5.129420
175,1970-01-01 00:00:00.000000175,WTG_07,wind,0.0,5.45,30.93,1.38,1.105,2.1445,0.1885,-3.719590
198,1970-01-01 00:00:00.000000198,WTG_07,wind,0.0,5.53,27.82,1.44,1.140,0.3830,0.1275,4.004678
199,1970-01-01 00:00:00.000000199,WTG_07,wind,0.0,6.38,28.25,1.21,1.521,0.4535,0.1615,4.458382


**Method 2 - Initial Findings

- **Window size:** 12 previous observations; minimum 6 historical readings.
- **Detection threshold:** Absolute modified deviation score greater than 3.5.
- **Potential anomalies detected:** 11 out of 240 observations (4.58%).
- **Distribution:** 5 solar observations and 6 wind observations.
- The flagged observations include repeated solar power deviations and consecutive wind power deviations.

**Note:** These are candidate anomalies identified relative to historical power behavior, not confirmed equipment faults. Detection consistency, threshold sensitivity, and feature leakage will be evaluated during the final comparison.

# Residual-Based Anomaly Detection

In [22]:
# Defining predictors separately for each asset
solar_features = [
    "irradiance_wm2",
    "temperature_c",
    "hour_sin",
    "hour_cos"
]

wind_features = [
    "wind_speed_ms",
    "temperature_c",
    "hour_sin",
    "hour_cos"
]

# Storing results separately from the original engineered dataset
residual_results = engg_df[
    ["timestamp_id", "asset_id", "asset_type", "power_mw"]
].copy()

residual_results["expected_power_mw"] = np.nan
residual_results["residual_mw"] = np.nan

# Fitting a separate baseline regression model for each asset
for asset_type, features in [
    ("solar", solar_features),
    ("wind", wind_features)
]:
    mask = engg_df["asset_type"].eq(asset_type)

    X = engg_df.loc[mask, features]
    y = engg_df.loc[mask, "power_mw"]

    # Fit only on rows with complete predictor and target values
    valid = X.notna().all(axis=1) & y.notna()

    model = LinearRegression()
    model.fit(X.loc[valid], y.loc[valid])

    # Calculate expected power and residuals
    predictions = model.predict(X.loc[valid])

    residual_results.loc[valid.index[valid], "expected_power_mw"] = predictions
    residual_results.loc[valid.index[valid], "residual_mw"] = (
        y.loc[valid] - predictions
    )

# Absolute residual measures the magnitude of the deviation
residual_results["absolute_residual_mw"] = (
    residual_results["residual_mw"].abs()
)

# Inspect the results
residual_results.head(10)


,timestamp_id,asset_id,asset_type,power_mw,expected_power_mw,residual_mw,absolute_residual_mw
0,1970-01-01 00:00:00.000000000,SOLAR_INV_01,solar,0.000,0.000028,-0.000028,0.000028
1,1970-01-01 00:00:00.000000001,SOLAR_INV_01,solar,0.000,0.000023,-0.000023,0.000023
2,1970-01-01 00:00:00.000000002,SOLAR_INV_01,solar,0.000,0.000010,-0.000010,0.000010
3,1970-01-01 00:00:00.000000003,SOLAR_INV_01,solar,0.000,-0.000011,0.000011,0.000011
4,1970-01-01 00:00:00.000000004,SOLAR_INV_01,solar,0.000,-0.000039,0.000039,0.000039
5,1970-01-01 00:00:00.000000005,SOLAR_INV_01,solar,0.000,-0.000071,0.000071,0.000071
6,1970-01-01 00:00:00.000000006,SOLAR_INV_01,solar,0.000,-0.000106,0.000106,0.000106
7,1970-01-01 00:00:00.000000007,SOLAR_INV_01,solar,0.393,0.393383,-0.000383,0.000383
8,1970-01-01 00:00:00.000000008,SOLAR_INV_01,solar,0.760,0.759893,0.000107,0.000107
9,1970-01-01 00:00:00.000000009,SOLAR_INV_01,solar,1.075,1.074723,0.000277,0.000277


In [24]:
# engg_df["timestamp_id"] = engg_df["timestamp_id"].astype("int64")

In [27]:
# print(engg_df["timestamp_id"].dtype)
# print(engg_df["timestamp_id"].head())

In [26]:

# Recover the integer values represented by the datetime-like IDs
recovered_ids = pd.to_datetime(
    engg_df["timestamp_id"]
).astype("int64")

print("First recovered IDs:", recovered_ids.head(10).tolist())
print("Are IDs sequential from zero:",
      recovered_ids.reset_index(drop=True).equals(
          pd.Series(range(len(recovered_ids)), dtype="int64")
      ))


First recovered IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Are IDs sequential from zero: True


In [28]:
engg_df["timestamp_id"] = recovered_ids

In [29]:
engg_df.iloc[:2]

,timestamp_id,asset_id,irradiance_wm2,wind_speed_ms,temperature_c,vibration_mm_s,power_mw,asset_type,hour_sin,hour_cos,...,wind_speed_ms_rolling_median,wind_speed_ms_rolling_mad,wind_speed_ms_modified_zscore,power_change_mw,high_vibration_flag,high_temperature_flag,method2_rolling_median,method2_rolling_mad,method2_deviation_score,method2_anomaly_flag
0,0,SOLAR_INV_01,0.0,6.57,26.26,1.37,0.0,solar,0.000000,1.000000,...,NaN,NaN,NaN,NaN,False,False,NaN,NaN,NaN,False
1,1,SOLAR_INV_01,0.0,6.72,24.89,1.49,0.0,solar,0.258819,0.965926,...,NaN,NaN,NaN,0.0,False,False,NaN,NaN,NaN,False


In [31]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_sets = {
    "solar": [
        "irradiance_wm2",
        "temperature_c",
        "hour_sin",
        "hour_cos"
    ],
    "wind": [
        "wind_speed_ms",
        "temperature_c",
        "hour_sin",
        "hour_cos"
    ]
}

evaluation_results = []
test_predictions = []

for asset_type, features in feature_sets.items():

    # Selecting one asset and order observations by sequential ID
    data = engg_df.loc[
        engg_df["asset_type"].eq(asset_type)
    ].sort_values("timestamp_id").copy()

    # Exclude rowing with missing predictors or target
    data = data.dropna(subset=features + ["power_mw"])

    # First 80% for training; last 20% for testing
    split_index = int(len(data) * 0.8)

    train = data.iloc[:split_index]
    test = data.iloc[split_index:]

    X_train = train[features]
    y_train = train["power_mw"]

    X_test = test[features]
    y_test = test["power_mw"]

    # Fitting only on training observations
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predicting unseen, later observations
    predictions = model.predict(X_test)
    residuals = y_test.to_numpy() - predictions

    evaluation_results.append({
        "asset_type": asset_type,
        "train_rows": len(train),
        "test_rows": len(test),
        "MAE_MW": mean_absolute_error(y_test, predictions),
        "RMSE_MW": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

    test_predictions.append(pd.DataFrame({
        "timestamp_id": test["timestamp_id"].to_numpy(),
        "asset_id": test["asset_id"].to_numpy(),
        "asset_type": asset_type,
        "actual_power_mw": y_test.to_numpy(),
        "expected_power_mw": predictions,
        "residual_mw": residuals,
        "absolute_residual_mw": np.abs(residuals)
    }))

evaluation_df = pd.DataFrame(evaluation_results)
residual_test_df = pd.concat(test_predictions, ignore_index=True)

print("Model evaluation:")
display(evaluation_df)

print("Test-set residuals:")
display(residual_test_df.head(10))


Model evaluation:


,asset_type,train_rows,test_rows,MAE_MW,RMSE_MW,R2
0,solar,96,24,0.000111,0.000160,1.000000
1,wind,96,24,0.027327,0.031658,0.997874


Test-set residuals:


,timestamp_id,asset_id,asset_type,actual_power_mw,expected_power_mw,residual_mw,absolute_residual_mw
0,96,SOLAR_INV_01,solar,0.000,0.000028,-0.000028,0.000028
1,97,SOLAR_INV_01,solar,0.000,0.000024,-0.000024,0.000024
2,98,SOLAR_INV_01,solar,0.000,0.000011,-0.000011,0.000011
3,99,SOLAR_INV_01,solar,0.000,-0.000010,0.000010,0.000010
4,100,SOLAR_INV_01,solar,0.000,-0.000039,0.000039,0.000039
5,101,SOLAR_INV_01,solar,0.000,-0.000071,0.000071,0.000071
6,102,SOLAR_INV_01,solar,0.000,-0.000105,0.000105,0.000105
7,103,SOLAR_INV_01,solar,0.393,0.393384,-0.000384,0.000384
8,104,SOLAR_INV_01,solar,0.760,0.759893,0.000107,0.000107
9,105,SOLAR_INV_01,solar,1.075,1.074722,0.000278,0.000278


In [32]:
# Inspecting the test residual distributions

# Summarizing the residuals separately for solar and wind before setting any thresholds
residual_summary = (
    residual_test_df
    .groupby("asset_type")["residual_mw"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        minimum="min",
        maximum="max"
    )
)

residual_summary["median_absolute_residual_mw"] = (
    residual_test_df
    .groupby("asset_type")["absolute_residual_mw"]
    .median()
)

residual_summary["max_absolute_residual_mw"] = (
    residual_test_df
    .groupby("asset_type")["absolute_residual_mw"]
    .max()
)

display(residual_summary)


,count,mean,median,std,minimum,maximum,median_absolute_residual_mw,max_absolute_residual_mw
asset_type,,,,,,,,
solar,24,-2.822280e-07,0.000010,0.000163,-0.000385,0.000278,0.000070,0.000385
wind,24,-1.500466e-02,-0.017283,0.028476,-0.062881,0.033994,0.028622,0.062881


Observations
- Solar: Residuals are very small, with a maximum absolute residual of approximately 0.000385 MW.
- Wind: Residuals are larger, with a maximum absolute residual of approximately 0.062881 MW. The mean residual is negative, suggesting that the model tends to overpredict wind power on these test observations.
The difference is plausible because the two technologies have different operating characteristics. However, we should not assume that the larger wind residuals are anomalies.

In [34]:
# Refitting to store the training residuals
feature_sets = {
    "solar": [
        "irradiance_wm2",
        "temperature_c",
        "hour_sin",
        "hour_cos"
    ],
    "wind": [
        "wind_speed_ms",
        "temperature_c",
        "hour_sin",
        "hour_cos"
    ]
}

thresholds = {}
residual_test_df = residual_test_df.copy()

for asset_type, features in feature_sets.items():

    data = engg_df.loc[
        engg_df["asset_type"].eq(asset_type)
    ].sort_values("timestamp_id").dropna(
        subset=features + ["power_mw"]
    )

    split_index = int(len(data) * 0.8)
    train = data.iloc[:split_index]
    test = data.iloc[split_index:]

    # Fit model using training observations only
    model = LinearRegression()
    model.fit(train[features], train["power_mw"])

    # Calculate training residuals
    train_predictions = model.predict(train[features])
    train_residuals = train["power_mw"].to_numpy() - train_predictions

    # Robust residual threshold
    residual_center = np.median(train_residuals)
    mad = np.median(np.abs(train_residuals - residual_center))
    robust_sigma = 1.4826 * mad

    # Fallback when MAD is zero
    if robust_sigma > 0:
        threshold = 3 * robust_sigma
    else:
        threshold = np.quantile(
            np.abs(train_residuals - residual_center), 0.99
        )

    thresholds[asset_type] = {
        "residual_center": residual_center,
        "mad": mad,
        "threshold_mw": threshold
    }

    # Apply the threshold to unseen test observations
    test_predictions = model.predict(test[features])
    test_residuals = test["power_mw"].to_numpy() - test_predictions

    test_mask = (
        residual_test_df["asset_type"].eq(asset_type)
    )

    # Match by timestamp_id to preserve row alignment
    test_ids = test["timestamp_id"].to_numpy()
    for timestamp_id, prediction, residual in zip(
        test_ids, test_predictions, test_residuals
    ):
        row_mask = (
            residual_test_df["asset_type"].eq(asset_type)
            & residual_test_df["timestamp_id"].eq(timestamp_id)
        )

        residual_test_df.loc[row_mask, "expected_power_mw"] = prediction
        residual_test_df.loc[row_mask, "residual_mw"] = residual
        residual_test_df.loc[row_mask, "absolute_residual_mw"] = abs(residual)

    # Flag deviations from the training residual center
    residual_test_df.loc[test_mask, "residual_anomaly"] = (
        residual_test_df.loc[test_mask, "residual_mw"]
        .sub(residual_center).abs() > threshold
    )

# Display thresholds
threshold_df = pd.DataFrame(thresholds).T
display(threshold_df)

# Display flagged test observations
flagged_residuals = residual_test_df.loc[
    residual_test_df["residual_anomaly"].eq(True)
].sort_values(["asset_type", "timestamp_id"])

display(flagged_residuals)


,residual_center,mad,threshold_mw
solar,0.000011,0.00006,0.000269
wind,0.001499,0.02563,0.113996


,timestamp_id,asset_id,asset_type,actual_power_mw,expected_power_mw,residual_mw,absolute_residual_mw,residual_anomaly
7,103,SOLAR_INV_01,solar,0.393,0.393384,-0.000384,0.000384,True
17,113,SOLAR_INV_01,solar,0.393,0.393385,-0.000385,0.000385,True


In [ ]:

# Inspecting the residual-based anomalies in the original engineered dataset(Validation)
flagged_ids = flagged_residuals["timestamp_id"].tolist()

columns_to_check = [
    "timestamp_id",
    "asset_id",
    "asset_type",
    "irradiance_wm2",
    "temperature_c",
    "wind_speed_ms",
    "vibration_mm_s",
    "power_mw"
]

display(
    engg_df.loc[
        engg_df["timestamp_id"].isin(flagged_ids),
        columns_to_check
    ].sort_values("timestamp_id")
)


,timestamp_id,asset_id,asset_type,irradiance_wm2,temperature_c,wind_speed_ms,vibration_mm_s,power_mw
103,103,SOLAR_INV_01,solar,207.1,29.56,6.30,1.32,0.393
113,113,SOLAR_INV_01,solar,207.1,37.39,8.94,1.41,0.393


Observations

Both observations have the same irradiance and power, but different temperatures. Their vibration readings are also relatively close, so vibration alone does not provide strong evidence of a mechanical issue.

The residual detector flags both because their prediction errors exceed the training-derived threshold. However, the very small threshold and unusually precise regression fit mean we should treat these as provisional flags, not confirmed anomalies.

# Isolation Forest Method

In [37]:

from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Select numerical features relevant to each technology
solar_features = [
    "irradiance_wm2",
    "temperature_c",
    "vibration_mm_s",
    "hour_sin",
    "hour_cos"
]

wind_features = [
    "wind_speed_ms",
    "temperature_c",
    "vibration_mm_s",
    "hour_sin",
    "hour_cos"
]

# Create a separate dataset for Isolation Forest
isolation_df = engg_df[
    ["timestamp_id", "asset_id", "asset_type", "power_mw"]
].copy()

# Store model-ready numerical features separately
solar_data = engg_df.loc[
    engg_df["asset_type"].eq("solar"),
    solar_features
].copy()

wind_data = engg_df.loc[
    engg_df["asset_type"].eq("wind"),
    wind_features
].copy()

# Inspect feature types and missing values before preprocessing
print("Solar feature information:")
display(solar_data.dtypes)
display(solar_data.isna().sum())

print("Wind feature information:")
display(wind_data.dtypes)
display(wind_data.isna().sum())


Solar feature information:


irradiance_wm2    float64
temperature_c     float64
vibration_mm_s    float64
hour_sin          float64
hour_cos          float64
dtype: object

irradiance_wm2    0
temperature_c     0
vibration_mm_s    0
hour_sin          0
hour_cos          0
dtype: int64

Wind feature information:


wind_speed_ms     float64
temperature_c     float64
vibration_mm_s    float64
hour_sin          float64
hour_cos          float64
dtype: object

wind_speed_ms     0
temperature_c     0
vibration_mm_s    0
hour_sin          0
hour_cos          0
dtype: int64

In [38]:
# Training separate Isolation Forest models, assumed conatmination is 0.05
models = {}
isolation_results = engg_df[
    ["timestamp_id", "asset_id", "asset_type", "power_mw"]
].copy()

isolation_results["anomaly_score"] = np.nan
isolation_results["isolation_anomaly"] = False

feature_sets = {
    "solar": solar_features,
    "wind": wind_features
}

for asset_type, features in feature_sets.items():

    # Select the relevant asset observations
    mask = engg_df["asset_type"].eq(asset_type)
    X = engg_df.loc[mask, features]

    # Train the model
    model = IsolationForest(
        n_estimators=200,
        contamination=0.05,
        random_state=42
    )
    model.fit(X)

    # Higher scores indicate more unusual observations
    scores = -model.decision_function(X)
    predictions = model.predict(X)

    # Store the model for later use
    models[asset_type] = model

    # Store results without modifying engg_df
    isolation_results.loc[X.index, "anomaly_score"] = scores
    isolation_results.loc[X.index, "isolation_anomaly"] = (
        predictions == -1
    )

# Display the detected observations
flagged_isolation = (
    isolation_results[
        isolation_results["isolation_anomaly"]
    ]
    .sort_values(["asset_type", "anomaly_score"], ascending=[True, False])
)

display(flagged_isolation)

print("Flagged observations by asset type:")
display(
    isolation_results.groupby("asset_type")["isolation_anomaly"]
    .sum()
    .rename("flagged_count")
    .to_frame()
)


,timestamp_id,asset_id,asset_type,power_mw,anomaly_score,isolation_anomaly
67,67,SOLAR_INV_01,solar,0.000,0.159431,True
69,69,SOLAR_INV_01,solar,0.000,0.136794,True
68,68,SOLAR_INV_01,solar,0.000,0.130520,True
109,109,SOLAR_INV_01,solar,1.468,0.017691,True
14,14,SOLAR_INV_01,solar,1.316,0.007582,True
105,105,SOLAR_INV_01,solar,1.075,0.002318,True
174,174,WTG_07,wind,0.711,0.163758,True
173,173,WTG_07,wind,0.726,0.153646,True
166,166,WTG_07,wind,2.618,0.046816,True
238,238,WTG_07,wind,0.077,0.033497,True


Flagged observations by asset type:


,flagged_count
asset_type,
solar,6
wind,6


In [43]:
print(engg_df.columns.tolist())

['timestamp_id', 'asset_id', 'irradiance_wm2', 'wind_speed_ms', 'temperature_c', 'vibration_mm_s', 'power_mw', 'asset_type', 'hour_sin', 'hour_cos', 'power_mw_rolling_median', 'power_mw_rolling_mad', 'power_mw_modified_zscore', 'vibration_mm_s_rolling_median', 'vibration_mm_s_rolling_mad', 'vibration_mm_s_modified_zscore', 'irradiance_wm2_rolling_median', 'irradiance_wm2_rolling_mad', 'irradiance_wm2_modified_zscore', 'wind_speed_ms_rolling_median', 'wind_speed_ms_rolling_mad', 'wind_speed_ms_modified_zscore', 'power_change_mw', 'high_vibration_flag', 'high_temperature_flag', 'method2_rolling_median', 'method2_rolling_mad', 'method2_deviation_score', 'method2_anomaly_flag']


In [44]:
# Inspecting the original telemetry for Isolation Forest flags
flagged_ids = flagged_isolation["timestamp_id"]

inspection_columns = [
    "timestamp_id",
    "asset_id",
    "asset_type",
    "irradiance_wm2",
    "wind_speed_ms",
    "temperature_c",
    "vibration_mm_s",
    "power_mw"
]

display(
    engg_df.loc[
        engg_df["timestamp_id"].isin(flagged_ids),
        inspection_columns
    ].sort_values(["asset_type", "timestamp_id"])
)

,timestamp_id,asset_id,asset_type,irradiance_wm2,wind_speed_ms,temperature_c,vibration_mm_s,power_mw
14,14,SOLAR_INV_01,solar,692.8,8.79,41.67,1.49,1.316
67,67,SOLAR_INV_01,solar,0.0,8.33,53.48,3.86,0.000
68,68,SOLAR_INV_01,solar,0.0,8.31,48.55,3.98,0.000
69,69,SOLAR_INV_01,solar,0.0,6.96,46.04,3.77,0.000
105,105,SOLAR_INV_01,solar,565.7,6.20,33.88,1.22,1.075
109,109,SOLAR_INV_01,solar,772.7,8.52,41.71,1.23,1.468
166,166,WTG_07,wind,0.0,8.82,26.26,1.49,2.618
173,173,WTG_07,wind,0.0,5.93,45.43,3.72,0.726
174,174,WTG_07,wind,0.0,5.87,46.88,3.82,0.711
184,184,WTG_07,wind,0.0,3.35,40.71,1.39,0.157


Observation

Irradiance = 0 and power = 0: consistent with normal solar shutdown when there is no sunlight.

Vibration = 3.77–3.98 mm/s: needs comparison with the inverter's normal vibration range; these values alone don't establish a fault.

Isolation Forest flagged them: this means their combined feature patterns look unusual to the model, not necessarily that the equipment is faulty.

In [ ]:
# Running the Isolation Forest again with 'power_mw' feature added

In [46]:

# Updated Isolation Forest feature sets
solar_features = [
    "irradiance_wm2",
    "temperature_c",
    "vibration_mm_s",
    "power_mw",
    "hour_sin",
    "hour_cos"
]

wind_features = [
    "wind_speed_ms",
    "temperature_c",
    "vibration_mm_s",
    "power_mw",
    "hour_sin",
    "hour_cos"
]

# Initialize result table
isolation_results = engg_df[
    ["timestamp_id", "asset_id", "asset_type", "power_mw"]
].copy()

isolation_results["anomaly_score"] = np.nan
isolation_results["isolation_anomaly"] = False

models = {}

feature_sets = {
    "solar": solar_features,
    "wind": wind_features
}

# Fit separate models for each asset type
for asset_type, features in feature_sets.items():
    mask = engg_df["asset_type"].eq(asset_type)
    X = engg_df.loc[mask, features]

    model = IsolationForest(
        n_estimators=200,
        contamination=0.05,
        random_state=42
    )

    model.fit(X)

    isolation_results.loc[X.index, "anomaly_score"] = (
        -model.decision_function(X)
    )
    isolation_results.loc[X.index, "isolation_anomaly"] = (
        model.predict(X) == -1
    )

    models[asset_type] = model

# Display flagged observations
flagged_isolation = (
    isolation_results.loc[
        isolation_results["isolation_anomaly"]
    ]
    .sort_values(
        ["asset_type", "anomaly_score"],
        ascending=[True, False]
    )
)

print("Flagged counts by asset type:")
display(
    isolation_results.groupby("asset_type")["isolation_anomaly"]
    .sum()
    .to_frame("flagged_count")
)

print("Flagged observations:")
display(flagged_isolation.to_string(index=False))


Flagged counts by asset type:


,flagged_count
asset_type,
solar,6
wind,6


Flagged observations:


' timestamp_id     asset_id asset_type  power_mw  anomaly_score  isolation_anomaly\n           67 SOLAR_INV_01      solar     0.000       0.134873               True\n           69 SOLAR_INV_01      solar     0.000       0.116301               True\n           68 SOLAR_INV_01      solar     0.000       0.112763               True\n          105 SOLAR_INV_01      solar     1.075       0.016868               True\n          109 SOLAR_INV_01      solar     1.468       0.009026               True\n           89 SOLAR_INV_01      solar     0.393       0.006989               True\n          174       WTG_07       wind     0.711       0.135251               True\n          173       WTG_07       wind     0.726       0.121060               True\n          166       WTG_07       wind     2.618       0.062258               True\n          238       WTG_07       wind     0.077       0.022665               True\n          218       WTG_07       wind     2.473       0.015827               True\n   

In [ ]:
# Comparing Isolation Forest with rolling median/MAD results

comparison_df = isolation_results[
    ["timestamp_id", "asset_id", "asset_type", "isolation_anomaly"]
].copy()

comparison_df = comparison_df.merge(
    engg_df[
        ["timestamp_id", "method2_anomaly_flag"]
    ],
    on="timestamp_id",
    how="left",
    validate="one_to_one"
)

comparison_df["method2_anomaly_flag"] = (
    comparison_df["method2_anomaly_flag"].fillna(False).astype(bool)
)

comparison_df["both_methods_flagged"] = (
    comparison_df["isolation_anomaly"]
    & comparison_df["method2_anomaly_flag"]
)

# Showing observations flagged by at least one of the two methods
display(
    comparison_df.loc[
        comparison_df["isolation_anomaly"]
        | comparison_df["method2_anomaly_flag"]
    ].sort_values(["asset_type", "timestamp_id"]).to_string(index=False)
)

print("Agreement summary:")
display(
    comparison_df.groupby("asset_type")[
        ["isolation_anomaly", "method2_anomaly_flag", "both_methods_flagged"]
    ].sum()
)


' timestamp_id     asset_id asset_type  isolation_anomaly  method2_anomaly_flag  both_methods_flagged\n           13 SOLAR_INV_01      solar              False                  True                 False\n           37 SOLAR_INV_01      solar              False                  True                 False\n           61 SOLAR_INV_01      solar              False                  True                 False\n           67 SOLAR_INV_01      solar               True                 False                 False\n           68 SOLAR_INV_01      solar               True                 False                 False\n           69 SOLAR_INV_01      solar               True                 False                 False\n           85 SOLAR_INV_01      solar              False                  True                 False\n           89 SOLAR_INV_01      solar               True                 False                 False\n          105 SOLAR_INV_01      solar               True                 False   

Agreement summary:


,isolation_anomaly,method2_anomaly_flag,both_methods_flagged
asset_type,,,
solar,6,5,1
wind,6,6,2


In [48]:

# Consolidated comparison of all four anomaly-detection methods

# Start with observation identifiers
comparison_all = engg_df[
    ["timestamp_id", "asset_id", "asset_type"]
].copy()

# Method 1: Rule-based detection
# Using the existing vibration and temperature rule flags
comparison_all["method1_rule_flag"] = (
    engg_df["high_vibration_flag"].fillna(False).astype(bool)
    | engg_df["high_temperature_flag"].fillna(False).astype(bool)
)

# Method 2: Rolling Median/MAD
comparison_all["method2_rolling_flag"] = (
    engg_df["method2_anomaly_flag"].fillna(False).astype(bool)
)

# Method 3: Residual-based detection
# residual_test_df contains the residual-based test observations
residual_flags = (
    residual_test_df[
        ["timestamp_id", "residual_anomaly"]
    ]
    .drop_duplicates(subset="timestamp_id")
    .set_index("timestamp_id")["residual_anomaly"]
)

comparison_all["method3_residual_flag"] = (
    comparison_all["timestamp_id"]
    .map(residual_flags)
    .fillna(False)
    .astype(bool)
)

# Method 4: Isolation Forest
isolation_flags = (
    isolation_results[
        ["timestamp_id", "isolation_anomaly"]
    ]
    .drop_duplicates(subset="timestamp_id")
    .set_index("timestamp_id")["isolation_anomaly"]
)

comparison_all["method4_isolation_flag"] = (
    comparison_all["timestamp_id"]
    .map(isolation_flags)
    .fillna(False)
    .astype(bool)
)

# Count how many methods flagged each observation
flag_columns = [
    "method1_rule_flag",
    "method2_rolling_flag",
    "method3_residual_flag",
    "method4_isolation_flag"
]

comparison_all["methods_flagged_count"] = (
    comparison_all[flag_columns].sum(axis=1)
)

# Display observations flagged by at least one method
flagged_comparison = (
    comparison_all[
        comparison_all["methods_flagged_count"] > 0
    ]
    .sort_values(
        ["methods_flagged_count", "asset_type", "timestamp_id"],
        ascending=[False, True, True]
    )
)

display(flagged_comparison.to_string(index=False))

# Summary: number of observations flagged by each method
print("Flag counts by method:")
display(
    comparison_all[flag_columns].sum().to_frame("flagged_count")
)

print("Agreement summary:")
display(
    comparison_all["methods_flagged_count"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_methods_flagging")
    .to_frame("observation_count")
)


' timestamp_id     asset_id asset_type  method1_rule_flag  method2_rolling_flag  method3_residual_flag  method4_isolation_flag  methods_flagged_count\n          173       WTG_07       wind               True                  True                  False                    True                      3\n          174       WTG_07       wind               True                  True                  False                    True                      3\n           67 SOLAR_INV_01      solar               True                 False                  False                    True                      2\n           68 SOLAR_INV_01      solar               True                 False                  False                    True                      2\n           69 SOLAR_INV_01      solar               True                 False                  False                    True                      2\n          109 SOLAR_INV_01      solar              False                  True                  Fal

Flag counts by method:


,flagged_count
method1_rule_flag,5
method2_rolling_flag,11
method3_residual_flag,2
method4_isolation_flag,12


Agreement summary:


,observation_count
number_of_methods_flagging,
0,218
1,16
2,4
3,2


Observation

The strongest agreement so far is for wind observations 173 and 174, which were flagged by three methods: rule-based, rolling median/MAD, and Isolation Forest.

Solar observations 67, 68, and 69 were flagged by two methods, but we already saw that their zero power corresponds to zero irradiance. They should not automatically be treated as faults.

In [50]:

# Investigation observations flagged by at least two methods

candidate_ids = comparison_all.loc[
    comparison_all["methods_flagged_count"] >= 2,
    "timestamp_id"
]

inspection_columns = [
    "timestamp_id",
    "asset_id",
    "asset_type",
    "irradiance_wm2",
    "wind_speed_ms",
    "temperature_c",
    "vibration_mm_s",
    "power_mw"
]

display(
    engg_df.loc[
        engg_df["timestamp_id"].isin(candidate_ids),
        inspection_columns
    ]
    .sort_values(["asset_type", "timestamp_id"])
    .to_string(index=False)
)


' timestamp_id     asset_id asset_type  irradiance_wm2  wind_speed_ms  temperature_c  vibration_mm_s  power_mw\n           67 SOLAR_INV_01      solar             0.0           8.33          53.48            3.86     0.000\n           68 SOLAR_INV_01      solar             0.0           8.31          48.55            3.98     0.000\n           69 SOLAR_INV_01      solar             0.0           6.96          46.04            3.77     0.000\n          109 SOLAR_INV_01      solar           772.7           8.52          41.71            1.23     1.468\n          173       WTG_07       wind             0.0           5.93          45.43            3.72     0.726\n          174       WTG_07       wind             0.0           5.87          46.88            3.82     0.711'

We have six observations flagged by at least two methods.
- Solar 67–69: Zero irradiance and zero power are consistent with normal behavior. However, the relatively high temperatures deserve context; we should not label them faulty based on Isolation Forest alone.
- Solar 109: Irradiance is 772.7 W/m² and power is 1.468 MW. 
- Wind 173–174: Wind speed is about 5.9 m/s, power is around 0.72 MW, and vibration is about 3.7–3.8 mm/s. These deserve closer investigation because three methods flagged them.

In [51]:

# Inspecting nearby observations around the strongest candidates

candidate_ids = [67, 68, 69, 109, 173, 174]
offset = 2

inspection_columns = [
    "timestamp_id",
    "asset_id",
    "asset_type",
    "irradiance_wm2",
    "wind_speed_ms",
    "temperature_c",
    "vibration_mm_s",
    "power_mw"
]

for timestamp_id in candidate_ids:
    candidate = engg_df.loc[
        engg_df["timestamp_id"].eq(timestamp_id)
    ]

    if candidate.empty:
        continue

    asset_id = candidate["asset_id"].iloc[0]
    nearby = engg_df.loc[
        engg_df["asset_id"].eq(asset_id)
        & engg_df["timestamp_id"].between(
            timestamp_id - offset,
            timestamp_id + offset
        ),
        inspection_columns
    ].sort_values("timestamp_id")

    print(f"\nObservations near timestamp_id {timestamp_id}:")
    display(nearby.to_string(index=False))



Observations near timestamp_id 67:


' timestamp_id     asset_id asset_type  irradiance_wm2  wind_speed_ms  temperature_c  vibration_mm_s  power_mw\n           65 SOLAR_INV_01      solar           207.1           8.55          37.69            1.43     0.393\n           66 SOLAR_INV_01      solar             0.0           6.98          36.37            1.44     0.000\n           67 SOLAR_INV_01      solar             0.0           8.33          53.48            3.86     0.000\n           68 SOLAR_INV_01      solar             0.0           8.31          48.55            3.98     0.000\n           69 SOLAR_INV_01      solar             0.0           6.96          46.04            3.77     0.000'


Observations near timestamp_id 68:


' timestamp_id     asset_id asset_type  irradiance_wm2  wind_speed_ms  temperature_c  vibration_mm_s  power_mw\n           66 SOLAR_INV_01      solar             0.0           6.98          36.37            1.44       0.0\n           67 SOLAR_INV_01      solar             0.0           8.33          53.48            3.86       0.0\n           68 SOLAR_INV_01      solar             0.0           8.31          48.55            3.98       0.0\n           69 SOLAR_INV_01      solar             0.0           6.96          46.04            3.77       0.0\n           70 SOLAR_INV_01      solar             0.0           8.15          28.14            1.47       0.0'


Observations near timestamp_id 69:


' timestamp_id     asset_id asset_type  irradiance_wm2  wind_speed_ms  temperature_c  vibration_mm_s  power_mw\n           67 SOLAR_INV_01      solar             0.0           8.33          53.48            3.86       0.0\n           68 SOLAR_INV_01      solar             0.0           8.31          48.55            3.98       0.0\n           69 SOLAR_INV_01      solar             0.0           6.96          46.04            3.77       0.0\n           70 SOLAR_INV_01      solar             0.0           8.15          28.14            1.47       0.0\n           71 SOLAR_INV_01      solar             0.0           7.89          25.87            1.22       0.0'

... (2 outputs left out)


Observations near timestamp_id 173:


' timestamp_id asset_id asset_type  irradiance_wm2  wind_speed_ms  temperature_c  vibration_mm_s  power_mw\n          171   WTG_07       wind             0.0           7.38          26.06            1.45     1.971\n          172   WTG_07       wind             0.0           7.38          23.42            1.29     1.972\n          173   WTG_07       wind             0.0           5.93          45.43            3.72     0.726\n          174   WTG_07       wind             0.0           5.87          46.88            3.82     0.711\n          175   WTG_07       wind             0.0           5.45          30.93            1.38     1.105'


Observations near timestamp_id 174:


' timestamp_id asset_id asset_type  irradiance_wm2  wind_speed_ms  temperature_c  vibration_mm_s  power_mw\n          172   WTG_07       wind             0.0           7.38          23.42            1.29     1.972\n          173   WTG_07       wind             0.0           5.93          45.43            3.72     0.726\n          174   WTG_07       wind             0.0           5.87          46.88            3.82     0.711\n          175   WTG_07       wind             0.0           5.45          30.93            1.38     1.105\n          176   WTG_07       wind             0.0           5.60          30.67            1.43     1.171'

Observation

Solar 67–69: Irradiance and power are both zero, but temperature and vibration suddenly rise and then return to normal at timestamp 70. The zero power is expected; the temperature and vibration changes are more suspicious and warrant investigation.

Solar 109: Power closely follows the surrounding irradiance readings. It is less obviously anomalous from these readings alone.

Wind 173–174: Wind speed drops, power drops sharply, temperature rises, and vibration jumps before returning toward normal at timestamp 175. This is a stronger candidate for an operational anomaly.

In [52]:

# Isolation Forest: contamination sensitivity analysis

contamination_values = [0.02, 0.05, 0.10]
sensitivity_results = []
sensitivity_flags = {}

for asset_type, features in {
    "solar": solar_features,
    "wind": wind_features
}.items():

    mask = engg_df["asset_type"].eq(asset_type)
    X = engg_df.loc[mask, features]

    for contamination in contamination_values:
        model = IsolationForest(
            n_estimators=200,
            contamination=contamination,
            random_state=42
        )

        model.fit(X)
        predictions = model.predict(X)

        flagged_ids = engg_df.loc[
            X.index[predictions == -1], "timestamp_id"
        ].tolist()

        sensitivity_flags[(asset_type, contamination)] = set(flagged_ids)

        sensitivity_results.append({
            "asset_type": asset_type,
            "contamination": contamination,
            "flagged_count": len(flagged_ids),
            "flagged_timestamp_ids": sorted(flagged_ids)
        })

sensitivity_df = pd.DataFrame(sensitivity_results)

display(sensitivity_df.to_string(index=False))

# Check selected candidates across contamination settings
candidate_ids = [67, 68, 69, 109, 173, 174]

candidate_stability = []

for asset_type in ["solar", "wind"]:
    for timestamp_id in candidate_ids:
        row = engg_df.loc[
            engg_df["timestamp_id"].eq(timestamp_id)
            & engg_df["asset_type"].eq(asset_type)
        ]

        if row.empty:
            continue

        result = {
            "asset_type": asset_type,
            "timestamp_id": timestamp_id
        }

        for contamination in contamination_values:
            flagged = timestamp_id in sensitivity_flags[
                (asset_type, contamination)
            ]
            result[f"flagged_at_{contamination:.2f}"] = flagged

        candidate_stability.append(result)

print("Stability of selected observations:")
display(pd.DataFrame(candidate_stability).to_string(index=False))


'asset_type  contamination  flagged_count                                        flagged_timestamp_ids\n     solar           0.02              3                                                 [67, 68, 69]\n     solar           0.05              6                                   [67, 68, 69, 89, 105, 109]\n     solar           0.10             12           [33, 41, 58, 60, 63, 67, 68, 69, 84, 89, 105, 109]\n      wind           0.02              3                                              [166, 173, 174]\n      wind           0.05              6                               [166, 173, 174, 216, 218, 238]\n      wind           0.10             12 [137, 159, 164, 166, 173, 174, 184, 212, 216, 218, 234, 238]'

Stability of selected observations:


'asset_type  timestamp_id  flagged_at_0.02  flagged_at_0.05  flagged_at_0.10\n     solar            67             True             True             True\n     solar            68             True             True             True\n     solar            69             True             True             True\n     solar           109            False             True             True\n      wind           173             True             True             True\n      wind           174             True             True             True'

Observation

At 0.02, each model flags 3 observations.

At 0.05, each flags 6 observations.

At 0.10, each flags 12 observations.

Solar timestamps 67–69 and wind timestamps 173–174 remain flagged at all three settings. These are stable candidates, although the solar zero-power readings still need to be interpreted in context.

Solar timestamp 109 is flagged only at 0.05 and 0.10, so its detection is more sensitive to the chosen setting.

## Final comparison summary of the four methods

In [53]:

flag_columns = [
    "method1_rule_flag",
    "method2_rolling_flag",
    "method3_residual_flag",
    "method4_isolation_flag"
]

print("Observations flagged by each method:")
display(
    comparison_all[flag_columns]
    .sum()
    .rename("flagged_count")
    .to_frame()
)

print("Observations flagged by multiple methods:")
display(
    comparison_all["methods_flagged_count"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_methods_flagging")
    .to_frame("observation_count")
)

print("Observations flagged by at least two methods:")
display(
    comparison_all.loc[
        comparison_all["methods_flagged_count"] >= 2
    ].sort_values(
        ["methods_flagged_count", "asset_type", "timestamp_id"],
        ascending=[False, True, True]
    ).to_string(index=False)
)


Observations flagged by each method:


,flagged_count
method1_rule_flag,5
method2_rolling_flag,11
method3_residual_flag,2
method4_isolation_flag,12


Observations flagged by multiple methods:


,observation_count
number_of_methods_flagging,
0,218
1,16
2,4
3,2


Observations flagged by at least two methods:


' timestamp_id     asset_id asset_type  method1_rule_flag  method2_rolling_flag  method3_residual_flag  method4_isolation_flag  methods_flagged_count\n          173       WTG_07       wind               True                  True                  False                    True                      3\n          174       WTG_07       wind               True                  True                  False                    True                      3\n           67 SOLAR_INV_01      solar               True                 False                  False                    True                      2\n           68 SOLAR_INV_01      solar               True                 False                  False                    True                      2\n           69 SOLAR_INV_01      solar               True                 False                  False                    True                      2\n          109 SOLAR_INV_01      solar              False                  True                  Fal

In [54]:
# Comparing all four methods on the same chronological test subset

# 1. Get the test observation IDs used by the residual-based method
test_ids = residual_test_df["timestamp_id"].unique()

# 2. Keep only these observations in the consolidated comparison
comparison_test = comparison_all[
    comparison_all["timestamp_id"].isin(test_ids)
].copy()

# 3. Count observations flagged by each method on this same subset
flag_columns = [
    "method1_rule_flag",
    "method2_rolling_flag",
    "method3_residual_flag",
    "method4_isolation_flag"
]

print("Flagged observations on the same test subset:")
display(
    comparison_test[flag_columns]
    .sum()
    .rename("flagged_count")
    .to_frame()
)

# 4. Count how many methods flagged each test observation
print("Number of test observations flagged by multiple methods:")
display(
    comparison_test["methods_flagged_count"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_methods_flagging")
    .to_frame("observation_count")
)

# 5. Inspect test observations flagged by at least two methods
print("Test observations flagged by at least two methods:")
display(
    comparison_test[
        comparison_test["methods_flagged_count"] >= 2
    ].sort_values(
        ["methods_flagged_count", "asset_type", "timestamp_id"],
        ascending=[False, True, True]
    )
)

Flagged observations on the same test subset:


,flagged_count
method1_rule_flag,0
method2_rolling_flag,1
method3_residual_flag,2
method4_isolation_flag,5


Number of test observations flagged by multiple methods:


,observation_count
number_of_methods_flagging,
0,41
1,6
2,1


Test observations flagged by at least two methods:


,timestamp_id,asset_id,asset_type,method1_rule_flag,method2_rolling_flag,method3_residual_flag,method4_isolation_flag,methods_flagged_count
109,109,SOLAR_INV_01,solar,False,True,False,True,2


In [56]:
# Inspecting observation 109 and nearby solar observations

nearby_solar = engg_df.loc[
    (engg_df["asset_id"] == "SOLAR_INV_01")
    & engg_df["timestamp_id"].between(107, 111),
    [
        "timestamp_id",
        "irradiance_wm2",
        "temperature_c",
        "vibration_mm_s",
        "power_mw",
        "method2_anomaly_flag",
        "high_vibration_flag",
        "high_temperature_flag"
    ]
].copy()

# Add Isolation Forest results
nearby_solar = nearby_solar.merge(
    isolation_results[
        ["timestamp_id", "isolation_anomaly", "anomaly_score"]
    ],
    on="timestamp_id",
    how="left"
)

display(nearby_solar)

,timestamp_id,irradiance_wm2,temperature_c,vibration_mm_s,power_mw,method2_anomaly_flag,high_vibration_flag,high_temperature_flag,isolation_anomaly,anomaly_score
0,107,772.7,37.65,1.48,1.468,False,False,False,False,-0.037080
1,108,800.0,39.37,1.29,1.520,False,False,False,False,-0.030285
2,109,772.7,41.71,1.23,1.468,True,False,False,True,0.009026
3,110,692.8,38.42,1.34,1.316,False,False,False,False,-0.041374
4,111,565.7,40.27,1.37,1.075,False,False,False,False,-0.047225


In [57]:
# Final model selection assessment

method_assessment = pd.DataFrame({
    "Method": [
        "Rule-based",
        "Rolling Median/MAD",
        "Residual-based",
        "Isolation Forest"
    ],
    "Flagged observations (full comparison)": [
        int(comparison_all["method1_rule_flag"].sum()),
        int(comparison_all["method2_rolling_flag"].sum()),
        int(comparison_all["method3_residual_flag"].sum()),
        int(comparison_all["method4_isolation_flag"].sum())
    ],
    "Interpretability": [
        "High",
        "High",
        "Medium",
        "Low to medium"
    ],
    "Main strength": [
        "Transparent domain rules",
        "Robust local deviation detection",
        "Detects unexpected power output",
        "Detects unusual feature combinations"
    ],
    "Main limitation": [
        "Depends on predefined thresholds",
        "Depends on window and threshold choices",
        "Depends on prediction quality; current flags cover test rows only",
        "Sensitive to contamination; no direct fault explanation"
    ],
    "Deployment suitability": [
        "Good as supporting checks",
        "Strong candidate for primary detector",
        "Useful complementary detector",
        "Better as a secondary detector for now"
    ]
})

display(method_assessment)

,Method,Flagged observations (full comparison),Interpretability,Main strength,Main limitation,Deployment suitability
0,Rule-based,5,High,Transparent domain rules,Depends on predefined thresholds,Good as supporting checks
1,Rolling Median/MAD,11,High,Robust local deviation detection,Depends on window and threshold choices,Strong candidate for primary detector
2,Residual-based,2,Medium,Detects unexpected power output,Depends on prediction quality; current flags c...,Useful complementary detector
3,Isolation Forest,12,Low to medium,Detects unusual feature combinations,Sensitive to contamination; no direct fault ex...,Better as a secondary detector for now


### Selected Model : Rolling Median/MAD

Rolling Median/MAD is selected as the primary anomaly-detection method because it is robust to extreme values, detects deviations from recent telemetry patterns, is interpretable, and is computationally efficient.

**Evidence from evaluation:**
- Flagged 11 observations, compared with 5 for rule-based detection and 12 for Isolation Forest (contamination = 0.05).
- Detected local deviations, including wind observations 173 and 174, which also showed sharp changes in temperature, vibration, wind speed, and power.
- Isolation Forest sensitivity testing showed that flagged observations varied with the contamination setting, while Rolling Median/MAD provides a more directly interpretable local baseline.

Since the dataset has no confirmed anomaly labels, these results support a practical model choice but do not prove superior detection accuracy. Solar and wind assets require appropriate operating baselines and thresholds, and detected anomalies should be treated as candidates for investigation rather than confirmed faults.